In [89]:
import pandapower as pp
import pandapower.shortcircuit as sc
net = pp.create_empty_network()

In [90]:
#create the high voltage bus and medium voltage bus, as well as the load bus

In [91]:
b_hv = pp.create_bus(net, vn_kv=138.0, name="HV Bus")
b_mv = pp.create_bus(net, vn_kv=12.47, name="MV Bus")
b_load = pp.create_bus(net, vn_kv=12.47, name="Load Bus")

In [92]:
#Create the slack bus and assign b_hv to it as the utility source. Include source strength parameters for fault analysis

In [93]:
pp.create_ext_grid(net, bus=b_hv, vm_pu=1.02, name="Utility source",
                    s_sc_max_mva=2500, rx_max=0.1,
                    s_sc_min_mva=2000, rx_min=0.1)

np.int64(0)

In [94]:
#Create the transformer utilizing b_hv and b_mv and their nominal voltage values. set the vk percent to 8.

In [95]:
pp.create_transformer_from_parameters(
    net, hv_bus=b_hv, lv_bus=b_mv,
    sn_mva=25.0, vn_hv_kv=138.0, vn_lv_kv=12.47,
    vkr_percent=0.5, vk_percent=8.0,
    pfe_kw=30.0, i0_percent=0.1,
    name="Substation XMFR")

np.int64(0)

In [96]:
#create the 3km line from lv side of transformer to load bus, with set reactance and resistive values

In [97]:
pp.create_line_from_parameters(
    net, from_bus=b_mv, to_bus=b_load, length_km=3.0,
    r_ohm_per_km=0.3, x_ohm_per_km=0.35, c_nf_per_km=10.0,
    max_i_ka=0.4, name="Feeder 1")

np.int64(0)

In [98]:
#create the load with real and reactive power parameters

In [99]:
pp.create_load(net, bus=b_load, p_mw=8.0, q_mvar=3.0, name="Distribution Load")
print(net)

This pandapower network includes the following parameter tables:
   - bus (3 elements)
   - load (1 element)
   - ext_grid (1 element)
   - line (1 element)
   - trafo (1 element)


In [100]:
#run short circuit solver

In [101]:
# maximum 3 phase fualt
sc.calc_sc(net, case="max", fault="3ph")
net.res_bus_sc

C:\Users\riley\miniconda3\envs\pcenv\Lib\site-packages\pandapower\build_branch.py:1590: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)
C:\Users\riley\miniconda3\envs\pcenv\Lib\site-packages\pandapower\build_branch.py:1590: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  power_station_unit = trafo_df.power_station_unit.fillna(False).values.astype(bool)


,ikss_ka,skss_mw,rk_ohm,xk_ohm
0,10.459244,2500.000000,0.833777,8.337775
1,14.026723,302.958616,0.037822,0.563334
2,4.243869,91.661941,0.937822,1.613334


In [ ]:
#Maximum fault current occurs when the line is shorted at bus1 (lv side of transformer). This tells us that the breaker we utilize
#must be rated high enough to handle 14 kA of current without the connections melting shut so that the breaker can actually trip.
#source strength here is the max threshold set by ext_grid in bus 0 2500 mva

In [102]:
net.line["endtemp_degree"] = 250   #model the feeder line when hot

In [103]:
# minimum 3 phase fault

In [104]:
sc.calc_sc(net, case="min", fault="3ph")
net.res_bus_sc

,ikss_ka,skss_mw,rk_ohm,xk_ohm
0,8.367395,2000.000000,0.947474,9.474744
1,12.514289,270.292042,0.038837,0.573995
2,3.000058,64.797271,1.766837,1.623995


In [ ]:
'''This shows the minimum fault assuming a source power of 2000 mva instead of 2500 mva, and also takes into account the increased
resistance in the feeder line given it's heated to 250 degrees celcius. In this case the minimum fault current is 3 kA when a short
occurs at bus 2 at the load, and that 3 kA is seen at every point on the feeder line all the way up to bus 1 (the lv transformer).
This 3 kA fault current becomes our min fault threshold.

Our load is set to 8 mw and 3 mvar which comes out to around 8.5 mva. Under normal conditions, current draw will be 8.5 mva/(sqrt(3)*12.47 kv)
Normal current draw of the load is about 400 A. Our pickup window for the relay is between 400 A and 3 kA, so we'll set the pickup current for
the relay as 550 A. At 550 A the relay timer would start, and it trips quicker the higher the current. When the relay starts to see
current loads closer to 3kA this represents the minimum short circuit if it happened with source at 2000 mva and a short at bus 2, so the relay must
trip quicker. at max fault conditions with source at 2500 mva, a short at bus 2 creates about 4.24 kA, and this is the fault current where the relay
must basically trip instantly. As the short circuit happens closer and closer to the transformer, current will keep increasing until a maximum of 
14 kA, which is why the breaker must be rated to handle this maximum event, but in theory the relay pretty much trips instantly in the range of 4.24
to 14 kA.

Transformer Maximum Through Fault Power = St_max = (transformer power rating S)/Vk_percent   St_max = 25mva/0.08 = 312.5 mva. This means if the 
source was infinite the transformer chokes down the power output to a maximum of 312.5 mva. Actual power output however would be 
S = (S_source * St_max)/(S_source + St_max), so S = (2500 * 312.5) / (2500 + 312.5) = 277.7 mva.'''